In [1]:
import pandas as pd
import numpy as np

# CHARGEMENT
df = pd.read_csv("D:\\rouge_file\\Analyse_du_performance_de_la_bourse_casablanca\\csv_clean\\Obligations_RENAMED.csv")
df = df[df['Type'] != 'Type'].drop_duplicates().copy()

print(f'Shape apres nettoyage basique : {df.shape}')
print(f'Nulls avant traitement        : {df.isnull().sum().sum()}')
print()

# SUPPRESSION LIGNE PARASITE RESIDUELLE
# Logique : la ligne avec Designation = "Designation" est un
# residue d en-tete OCR mal filtre — aucune donnee exploitable.
avant = len(df)
df = df[df['Designation'] != 'Désignation']
df = df[df['Statut'] != 'Statut']
print(f'Lignes parasites supprimees : {avant - len(df)}')
print()

# CORRECTION MATURITE "**" — CFG Bank
# Logique : la Maturite de l obligation CFG Bank est mal lue
# par l OCR — elle affiche "**" au lieu d un nombre.
# En lisant la designation "23DEC2021 4.69% IND 100K CFG"
# et en consultant les donnees de marche, cette obligation
# a une maturite indicielle (IND) sans echeance fixe.
# On remplace "**" par NaN puis on laisse tel quel —
# une obligation indicielle n a pas de maturite standard.
avant = (df['Maturité'] == '**').sum()
df['Maturité'] = df['Maturité'].replace('**', np.nan)
print(f'[NaN] {"Maturite CFG Bank":25s} : {avant} valeurs corrompues -> remplacees par NaN (obligation indicielle sans maturite fixe)')
print()

# GROUPE 1 — INFOS STABLES DE L OBLIGATION (< 1%)
# Forward Fill + Backward Fill par Designation
# Logique : toutes les caracteristiques d une obligation
# (emetteur, date emission, taux facial, type amortissement)
# sont fixees a l emission et ne changent jamais.
# Si une valeur manque pour une seance, on prend celle
# de la seance precedente pour la meme obligation.
df = df.sort_values(['Designation', 'Date Emission'])

cols_ffill = ['Emetteur', 'Date Emission', 'Maturité', 'Taux facial',
              'Type amort.', 'Date dernier coupon', 'Cours Réf %']

for col in cols_ffill:
    if col in df.columns:
        avant = df[col].isnull().sum()
        df[col] = df.groupby('Designation')[col].transform(
            lambda x: x.ffill().bfill()
        )
        apres = df[col].isnull().sum()
        print(f'[FFILL] {col:25s} : {avant} nulls -> {apres} nulls restants')

print()

# RAPPORT FINAL
print('RAPPORT FINAL')
print(f'Shape finale         : {df.shape}')
print(f'Nulls restants total : {df.isnull().sum().sum()}')
nulls_restants = df.isnull().sum()
reste = nulls_restants[nulls_restants > 0]
if len(reste) > 0:
    print('\nDetail nulls restants :')
    print(reste.to_string())
else:
    print('Aucun null restant — tableau 100% propre !')

df.to_csv('Obligations_FINAL.csv', index=False, encoding='utf-8')
print(f'\nExporte : Obligations_FINAL.csv — {len(df):,} lignes')

Shape apres nettoyage basique : (15791, 18)
Nulls avant traitement        : 8

Lignes parasites supprimees : 1

[NaN] Maturite CFG Bank         : 2281 valeurs corrompues -> remplacees par NaN (obligation indicielle sans maturite fixe)

[FFILL] Emetteur                  : 0 nulls -> 0 nulls restants
[FFILL] Date Emission             : 0 nulls -> 0 nulls restants


C:\Users\poste\AppData\Local\Temp\ipykernel_2820\903668231.py:50: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.ffill().bfill()


[FFILL] Maturité                  : 2281 nulls -> 2281 nulls restants
[FFILL] Taux facial               : 0 nulls -> 0 nulls restants
[FFILL] Type amort.               : 0 nulls -> 0 nulls restants
[FFILL] Date dernier coupon       : 1 nulls -> 0 nulls restants
[FFILL] Cours Réf %               : 0 nulls -> 0 nulls restants

RAPPORT FINAL
Shape finale         : (15790, 18)
Nulls restants total : 2281

Detail nulls restants :
Maturité    2281

Exporte : Obligations_FINAL.csv — 15,790 lignes
